# Data Collection

In [1]:
import requests
import time
import pandas as pd
from pathlib import Path

In [2]:
RAW_DIR  = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://api.jolpi.ca/ergast/f1'
SEASONS  = [2021, 2022, 2023, 2024, 2025]

print("Directories ready")
print("Seasons to collect:", SEASONS)

Directories ready
Seasons to collect: [2021, 2022, 2023, 2024, 2025]


In [4]:
def fetch_json(endpoint):
    url      = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, timeout=30)

    if response.status_code == 429:
        print(f"waiting 3s then retrying")
        time.sleep(3)
        response = requests.get(url, timeout=30)

    response.raise_for_status()
    time.sleep(0.35)
    return response.json()


def get_round_numbers(season):
    data  = fetch_json(f'{season}.json')
    races = data['MRData']['RaceTable']['Races']
    return [int(r['round']) for r in races]

## Quick test

In [5]:
rounds_2024 = get_round_numbers(2024)
print(f"2024: {len(rounds_2024)} rounds")
print(rounds_2024)

2024: 24 rounds
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


## Row Builders


In [6]:
def build_race_rows(data):
    rows = []
    for race in data['MRData']['RaceTable']['Races']:
        for result in race.get('Results', []):
            driver      = result['Driver']
            constructor = result['Constructor']
            rows.append({
                'season'          : int(race['season']),
                'round'           : int(race['round']),
                'race_name'       : race['raceName'],
                'date'            : race.get('date'),
                'driver_id'       : driver['driverId'],
                'driver_code'     : driver.get('code', ''),
                'driver_name'     : f"{driver['givenName']} {driver['familyName']}",
                'constructor'     : constructor['name'],
                'grid'            : pd.to_numeric(result.get('grid'),     errors='coerce'),
                'finish_position' : pd.to_numeric(result.get('position'), errors='coerce'),
                'points'          : pd.to_numeric(result.get('points'),   errors='coerce'),
                'status'          : result.get('status', '')
            })
    return rows


def build_quali_rows(data):
    rows = []
    for race in data['MRData']['RaceTable']['Races']:
        for result in race.get('QualifyingResults', []):
            driver      = result['Driver']
            constructor = result['Constructor']
            rows.append({
                'season'         : int(race['season']),
                'round'          : int(race['round']),
                'race_name'      : race['raceName'],
                'date'           : race.get('date'),
                'driver_id'      : driver['driverId'],
                'driver_code'    : driver.get('code', ''),
                'driver_name'    : f"{driver['givenName']} {driver['familyName']}",
                'constructor'    : constructor['name'],
                'quali_position' : pd.to_numeric(result.get('position'), errors='coerce'),
                'q1'             : result.get('Q1'),
                'q2'             : result.get('Q2'),
                'q3'             : result.get('Q3')
            })
    return rows

In [7]:
test_race  = fetch_json("2024/1/results.json")
test_quali = fetch_json("2024/1/qualifying.json")

race_sample  = build_race_rows(test_race)
quali_sample = build_quali_rows(test_quali)

print(f"Race rows from round 1: {len(race_sample)}")
print(f"Quali rows from round 1: {len(quali_sample)}")
print("\nFirst race row:")
print(race_sample[0])

Race rows from round 1: 20
Quali rows from round 1: 20

First race row:
{'season': 2024, 'round': 1, 'race_name': 'Bahrain Grand Prix', 'date': '2024-03-02', 'driver_id': 'max_verstappen', 'driver_code': 'VER', 'driver_name': 'Max Verstappen', 'constructor': 'Red Bull', 'grid': 1, 'finish_position': 1, 'points': 26, 'status': 'Finished'}
